In [ ]:
#Peak coordinates for grouped PeakMatrix are in rowData(se_sub), not rowRanges(se_sub).
library(SummarizedExperiment)
library(edgeR)
library(dplyr)
library(tibble)

atac_dir <- "/path/to/nature_study/ATAC_donor_level"
outdir_da <- file.path(atac_dir, "donor_aware_DA")
dir.create(outdir_da, showWarnings = FALSE, recursive = TRUE)

se_peak_donor <- readRDS(file.path(atac_dir, "PeakMatrix_rnaCelltypeByDonor_SE.rds"))

coldata <- as.data.frame(colData(se_peak_donor)) %>%
  rownames_to_column("group_id") %>%
  mutate(
    rna_celltype = sub("__.*$", "", group_id),
    donor = sub("^.*__", "", group_id),
    condition = ifelse(grepl("^TS21", donor), "Ts21", "disomy")
  )

run_atac_donor_DA <- function(se, coldata, celltype_keep, out_prefix, outdir) {

  cols_keep <- coldata$group_id[coldata$rna_celltype == celltype_keep]
  stopifnot(length(cols_keep) > 0)

  se_sub <- se[, cols_keep]
  meta_sub <- coldata %>%
    filter(group_id %in% cols_keep) %>%
    mutate(condition = factor(condition, levels = c("disomy", "Ts21"))) %>%
    arrange(match(group_id, colnames(se_sub)))

  counts <- as.matrix(assay(se_sub))

  y <- DGEList(counts = counts, samples = meta_sub)
  keep <- filterByExpr(y, group = meta_sub$condition)
  y <- y[keep, , keep.lib.sizes = FALSE]
  y <- calcNormFactors(y)

  design <- model.matrix(~ condition, data = meta_sub)
  rownames(design) <- meta_sub$group_id

  y <- estimateDisp(y, design)
  fit <- glmQLFit(y, design)
  qlf <- glmQLFTest(fit, coef = "conditionTs21")

  tab <- topTags(qlf, n = Inf, sort.by = "PValue")$table %>%
    rownames_to_column("peak_id")

  rr <- as.data.frame(rowData(se_sub)) %>%
    rownames_to_column("peak_id")

  tab <- left_join(tab, rr, by = "peak_id")

  coord_cols <- intersect(c("seqnames", "start", "end", "idx"), colnames(tab))
  other_cols <- setdiff(colnames(tab), c("peak_id", coord_cols))
  tab <- tab[, c("peak_id", coord_cols, other_cols), drop = FALSE]

  write.table(
    tab,
    file = file.path(outdir, paste0(out_prefix, "_edgeR_DA.tsv")),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )

  saveRDS(
    list(y = y, design = design, fit = fit, qlf = qlf, results = tab),
    file.path(outdir, paste0(out_prefix, "_edgeR_DA.rds"))
  )

  tab
}

da_hsc_tab <- run_atac_donor_DA(
  se = se_peak_donor,
  coldata = coldata,
  celltype_keep = "HSC_MPP",
  out_prefix = "HSC_MPP_Ts21_vs_disomy",
  outdir = outdir_da
)

dim(da_hsc_tab)
head(da_hsc_tab)
sum(da_hsc_tab$FDR <= 0.05, na.rm = TRUE)
sum(da_hsc_tab$FDR <= 0.05 & da_hsc_tab$logFC < 0, na.rm = TRUE)
sum(da_hsc_tab$FDR <= 0.05 & da_hsc_tab$logFC > 0, na.rm = TRUE)